In [1]:

from google import genai

import os

from dotenv import load_dotenv

In [2]:
load_dotenv()

client = genai.Client(
    api_key=os.getenv("GOOGLE_API_KEY")
)

In [3]:

import faiss

import numpy as np

In [4]:

docs = [
    "Our company policy states that all employees must complete security training annually.",
    "The employee onboarding process takes approximately two weeks.",
    "You can reset your company email password via the IT support portal.",
    "Remote work is allowed up to 3 days per week with manager approval."
]


In [5]:

result = client.models.embed_content(
    model="gemini-embedding-001",
    contents=docs
)


In [6]:

doc_embeddings = np.array(
    [embedding.values for embedding in result.embeddings],
    dtype="float32"
)

print("Embedding Shape:", doc_embeddings.shape)


Embedding Shape: (4, 3072)


In [7]:

dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(doc_embeddings)

print("Documents stored in FAISS:", index.ntotal)


Documents stored in FAISS: 4


In [8]:
def embed_question(question):

    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=[question]
    )

    q_embedding = np.array(
        [result.embeddings[0].values],
        dtype="float32"
    )

    return q_embedding


In [9]:
def search_faiss(q_embedding, k=2):

    D, I = index.search(q_embedding, k)

    return I[0]



In [10]:
def retrieve_documents(indices):

    retrieved_docs = [docs[i] for i in indices]

    return retrieved_docs


In [11]:
def create_context(retrieved_docs):

    context = "\n\n".join(retrieved_docs)

    return context


In [12]:
def create_prompt(question, context):

    prompt = f"""
You are a helpful company assistant.

Answer the question ONLY using the context below.

If the answer is not available in the context,
say "I don't know based on the provided documents."

CONTEXT:
{context}

QUESTION:
{question}

Give a concise and accurate answer.
"""

    return prompt


In [13]:
def generate_answer(prompt):

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text


In [14]:

def answer_question(question):

    #  Question → Embedding
    q_embedding = embed_question(question)

    #  FAISS Search
    indices = search_faiss(q_embedding, k=2)

    #  Retrieve Documents
    retrieved_docs = retrieve_documents(indices)

    #  Create Context
    context = create_context(retrieved_docs)

    #  Create Prompt
    prompt = create_prompt(question, context)

    # Generate Answer
    answer = generate_answer(prompt)

    return answer


In [15]:

question = "How do I reset my email password?"

answer = answer_question(question)

print("\nANSWER:")

print(answer)


ANSWER:
You can reset your company email password via the IT support portal.
